# Data Mining_2_Module_3
## 05 — Shapelets vs Motifs and Discords

This notebook compares the **10 most discriminative shapelets** from the selected `RandomShapeletTransform + LogisticRegression` model against:

1. the globally strongest 10 motif groups and 10 discords for each window (`w=6`, `w=12`);
2. all extracted motif occurrences and discords.

Because shapelets and Matrix Profile patterns come from different preprocessing scales, every compared local subsequence is **z-normalized locally** before Euclidean distance is computed.

In [1]:
from pathlib import Path
import warnings, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/Users/sarabiiavasco/Codice/DATA MINING/PROGETTO DM2"),
    Path("/Users/sarabiiavasco/Codice/DATA MINING/Progetto DM2"),
    Path("/Users/sarabiiavasco/Codice/DATA MINING/progetto dm2"),
]

def is_project_dir(path):
    return path.exists() and (
        (path / "data").exists()
        or (path / "artifacts").exists()
        or any(path.glob("CMI_timeseries_dataset*.pkl"))
    )

PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if is_project_dir(p)), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Project folder not found. Update PROJECT_CANDIDATES with your local project path."
    )

TS_PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "timeseries"
MODULE3_DIR = PROJECT_DIR / "artifacts" / "module3"

TS_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODULE3_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Processed time-series directory:", TS_PROCESSED_DIR)
print("Module 3 artifacts:", MODULE3_DIR)

Project directory: /Users/sarabiiavasco/Codice/DATA MINING/PROGETTO DM2
Processed time-series directory: /Users/sarabiiavasco/Codice/DATA MINING/PROGETTO DM2/data/processed/timeseries
Module 3 artifacts: /Users/sarabiiavasco/Codice/DATA MINING/PROGETTO DM2/artifacts/module3


In [2]:
DATA_PATH = TS_PROCESSED_DIR / "CMI_timeseries_dataset_preproc.pkl"
MOTIF_DIR = MODULE3_DIR / "motifs_discords"
CLASS_DIR = MODULE3_DIR / "classification"
COMPARE_DIR = MODULE3_DIR / "shapelet_motif_discord_comparison"
COMPARE_DIR.mkdir(parents=True,exist_ok=True)

required = [
    DATA_PATH,
    MOTIF_DIR/"motif_groups_df.pkl",
    MOTIF_DIR/"motif_occurrences_df.pkl",
    MOTIF_DIR/"discords_df.pkl",
    CLASS_DIR/"shapelet_metadata_top10.pkl",
]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Run notebooks 01, 02 and 04 first. Missing: " + ", ".join(map(str,missing))
    )

df = pd.read_pickle(DATA_PATH)
motif_groups_df = pd.read_pickle(MOTIF_DIR/"motif_groups_df.pkl")
motif_occurrences_df = pd.read_pickle(MOTIF_DIR/"motif_occurrences_df.pkl")
discords_df = pd.read_pickle(MOTIF_DIR/"discords_df.pkl")
shapelet_metadata_df = pd.read_pickle(CLASS_DIR/"shapelet_metadata_top10.pkl")

print("Time series:",len(df))
print("Motif groups:",motif_groups_df.shape)
print("Motif occurrences:",motif_occurrences_df.shape)
print("Discords:",discords_df.shape)
print("Selected shapelets:",shapelet_metadata_df.shape)
display(shapelet_metadata_df[["shapelet_index","shapelet_length","abs_lr_coefficient"]])

Time series: 4258
Motif groups: (23074, 9)
Motif occurrences: (71111, 15)
Discords: (25536, 14)
Selected shapelets: (10, 14)


,shapelet_index,shapelet_length,abs_lr_coefficient
0,38,33,1.022874
1,15,15,0.954442
2,30,35,0.850050
3,21,14,0.846139
4,26,24,0.845975
5,41,15,0.791571
6,49,12,0.787662
7,92,23,0.773508
8,0,23,0.770118
9,59,15,0.755225


In [3]:
# ============================================================
# GLOBAL TOP MOTIFS AND DISCORDS
# ============================================================

import numpy as np
import pandas as pd

TOP_K = 10
WINDOWS = [6, 12]


# ============================================================
# MOTIF GROUPS
# motif_groups_df already contains n_occurrences
# ============================================================

motif_groups_ranked_df = (
    motif_groups_df
    .copy()
)

# Safety check
if "n_occurrences" not in motif_groups_ranked_df.columns:
    raise KeyError(
        "Column 'n_occurrences' not found in motif_groups_df."
    )


# ------------------------------------------------------------
# TOP 10 MOTIF GROUPS PER WINDOW LENGTH
# Lowest motif distance = strongest motif
# ------------------------------------------------------------

top_motif_groups = {}

for w in WINDOWS:

    top_motif_groups[w] = (
        motif_groups_ranked_df[
            motif_groups_ranked_df["w"] == w
        ]
        .sort_values(
            "motif_distance",
            ascending=True
        )
        .head(TOP_K)
        .copy()
        .reset_index(drop=True)
    )

    top_motif_groups[w][
        "global_motif_rank"
    ] = np.arange(
        1,
        len(top_motif_groups[w]) + 1
    )


# ============================================================
# TOP 10 DISCORDS PER WINDOW LENGTH
# Highest discord score = strongest discord
# ============================================================

top_discords = {}

for w in WINDOWS:

    top_discords[w] = (
        discords_df[
            discords_df["w"] == w
        ]
        .sort_values(
            "discord_score",
            ascending=False
        )
        .head(TOP_K)
        .copy()
        .reset_index(drop=True)
    )

    top_discords[w][
        "global_discord_rank"
    ] = np.arange(
        1,
        len(top_discords[w]) + 1
    )


print("Top motif groups created successfully.")
print("Top discords created successfully.")

Top motif groups created successfully.
Top discords created successfully.


In [4]:
# ============================================================
# TOP 3 MOTIF GROUPS:
# NUMBER OF OCCURRENCES
# ============================================================

print("=" * 80)
print("TOP 3 MOTIF GROUPS AND NUMBER OF OCCURRENCES")
print("=" * 80)


for w in WINDOWS:

    print(f"\nWINDOW LENGTH w = {w}")
    print("-" * 80)

    top3 = (
        top_motif_groups[w]
        .head(3)
        .copy()
    )

    display(
        top3[
            [
                "global_motif_rank",
                "ts_index",
                "motif_rank",
                "motif_distance",
                "n_occurrences"
            ]
        ]
    )

    print(
        "Total occurrences across the top 3 motif groups:",
        top3["n_occurrences"].sum()
    )

TOP 3 MOTIF GROUPS AND NUMBER OF OCCURRENCES

WINDOW LENGTH w = 6
--------------------------------------------------------------------------------


,global_motif_rank,ts_index,motif_rank,motif_distance,n_occurrences
0,1,1229,1,2.220446e-16,2
1,2,3513,1,2.220446e-16,2
2,3,3586,1,1.168499e-04,2


Total occurrences across the top 3 motif groups: 6

WINDOW LENGTH w = 12
--------------------------------------------------------------------------------


,global_motif_rank,ts_index,motif_rank,motif_distance,n_occurrences
0,1,142,1,0.015359,2
1,2,3551,1,0.044023,2
2,3,1987,1,0.090686,2


Total occurrences across the top 3 motif groups: 6


In [5]:
# ============================================================
# SHAPE COMPARISON UTILITIES
# ============================================================

def z_normalize(x, eps=1e-10):
    """
    Local z-normalization.
    """
    x = np.asarray(
        x,
        dtype=float
    ).ravel()

    if not np.all(
        np.isfinite(x)
    ):
        return None

    std = np.std(x)

    if std <= eps:
        return None

    return (
        x - np.mean(x)
    ) / std


def znorm_euclidean(a, b):
    """
    Euclidean distance between two sequences
    of equal length after local z-normalization.
    """

    a = z_normalize(a)
    b = z_normalize(b)

    if (
        a is None
        or b is None
    ):
        return np.nan

    if len(a) != len(b):
        raise ValueError(
            "Sequences must have equal length."
        )

    return np.linalg.norm(
        a - b
    )


def get_shapelet_windows(
    shapelet_values,
    w
):
    """
    Extract all valid length-w windows
    contained inside a shapelet.
    """

    x = np.asarray(
        shapelet_values,
        dtype=float
    ).ravel()

    if len(x) < w:
        return []

    windows = []

    for start in range(
        len(x) - w + 1
    ):

        subsequence = x[
            start:start + w
        ]

        subsequence_z = z_normalize(
            subsequence
        )

        if subsequence_z is not None:

            windows.append({
                "shapelet_window_start":
                    start,

                "shapelet_window_end":
                    start + w,

                "values":
                    subsequence_z
            })

    return windows

In [6]:
# ============================================================
# OCCURRENCES OF THE GLOBAL TOP MOTIF GROUPS
# ============================================================

top_motif_occurrences = {}


for w in WINDOWS:

    selected_groups = (
        top_motif_groups[w][
            [
                "ts_index",
                "w",
                "motif_rank",
                "global_motif_rank",
                "motif_distance",
                "n_occurrences"
            ]
        ]
    )

    top_motif_occurrences[w] = (
        motif_occurrences_df
        .merge(
            selected_groups,
            on=[
                "ts_index",
                "w",
                "motif_rank"
            ],
            how="inner"
        )
        .copy()
    )


    print(
        f"w={w}: "
        f"{len(top_motif_groups[w])} top motif groups, "
        f"{len(top_motif_occurrences[w])} total occurrences"
    )

w=6: 10 top motif groups, 20 total occurrences
w=12: 10 top motif groups, 21 total occurrences


In [7]:
# ============================================================
# SHAPELETS vs GLOBAL TOP MOTIFS AND DISCORDS
# ============================================================

comparison_records = []


for _, shp in shapelet_metadata_df.iterrows():

    shapelet_index = int(
        shp["shapelet_index"]
    )

    shapelet_values = np.asarray(
        shp["shapelet_values"],
        dtype=float
    ).ravel()

    shapelet_length = len(
        shapelet_values
    )


    for w in WINDOWS:

        # Shapelet too short for this scale
        if shapelet_length < w:
            continue


        shapelet_windows = (
            get_shapelet_windows(
                shapelet_values,
                w
            )
        )


        # ====================================================
        # SHAPELET vs TOP MOTIF GROUPS
        # ====================================================

        for _, motif in (
            top_motif_occurrences[w]
            .iterrows()
        ):

            motif_ts_index = int(
                motif["ts_index"]
            )

            motif_start = int(
                motif["motif_start"]
            )

            motif_end = int(
                motif["motif_end"]
            )

            motif_values = (
                df[motif_ts_index][
                    "enmo"
                ]
                .to_numpy(
                    dtype=float
                )[
                    motif_start:motif_end
                ]
            )

            if len(
                motif_values
            ) != w:
                continue


            for sw in shapelet_windows:

                distance = (
                    znorm_euclidean(
                        sw["values"],
                        motif_values
                    )
                )

                if not np.isfinite(
                    distance
                ):
                    continue


                comparison_records.append({
                    "shapelet_index":
                        shapelet_index,

                    "shapelet_length":
                        shapelet_length,

                    "w":
                        w,

                    "pattern_type":
                        "motif",

                    "global_pattern_rank":
                        int(
                            motif[
                                "global_motif_rank"
                            ]
                        ),

                    "pattern_ts_index":
                        motif_ts_index,

                    "pattern_start":
                        motif_start,

                    "pattern_end":
                        motif_end,

                    "motif_group_rank":
                        int(
                            motif[
                                "motif_rank"
                            ]
                        ),

                    "motif_group_occurrences":
                        int(
                            motif[
                                "n_occurrences"
                            ]
                        ),

                    "shapelet_window_start":
                        sw[
                            "shapelet_window_start"
                        ],

                    "shapelet_window_end":
                        sw[
                            "shapelet_window_end"
                        ],

                    "distance":
                        distance
                })


        # ====================================================
        # SHAPELET vs TOP DISCORDS
        # ====================================================

        for _, discord in (
            top_discords[w]
            .iterrows()
        ):

            discord_ts_index = int(
                discord["ts_index"]
            )

            discord_start = int(
                discord[
                    "discord_start"
                ]
            )

            discord_end = int(
                discord[
                    "discord_end"
                ]
            )

            discord_values = (
                df[discord_ts_index][
                    "enmo"
                ]
                .to_numpy(
                    dtype=float
                )[
                    discord_start:discord_end
                ]
            )

            if len(
                discord_values
            ) != w:
                continue


            for sw in shapelet_windows:

                distance = (
                    znorm_euclidean(
                        sw["values"],
                        discord_values
                    )
                )

                if not np.isfinite(
                    distance
                ):
                    continue


                comparison_records.append({
                    "shapelet_index":
                        shapelet_index,

                    "shapelet_length":
                        shapelet_length,

                    "w":
                        w,

                    "pattern_type":
                        "discord",

                    "global_pattern_rank":
                        int(
                            discord[
                                "global_discord_rank"
                            ]
                        ),

                    "pattern_ts_index":
                        discord_ts_index,

                    "pattern_start":
                        discord_start,

                    "pattern_end":
                        discord_end,

                    "motif_group_rank":
                        np.nan,

                    "motif_group_occurrences":
                        np.nan,

                    "shapelet_window_start":
                        sw[
                            "shapelet_window_start"
                        ],

                    "shapelet_window_end":
                        sw[
                            "shapelet_window_end"
                        ],

                    "distance":
                        distance
                })


top_pattern_comparison_df = pd.DataFrame(
    comparison_records
)


print(
    "Total comparisons:",
    len(
        top_pattern_comparison_df
    )
)

display(
    top_pattern_comparison_df.head()
)

Total comparisons: 7839


,shapelet_index,shapelet_length,w,pattern_type,global_pattern_rank,pattern_ts_index,pattern_start,pattern_end,motif_group_rank,motif_group_occurrences,shapelet_window_start,shapelet_window_end,distance
0,38,33,6,motif,6,555,2,8,1.0,2.0,0,6,4.354799
1,38,33,6,motif,6,555,2,8,1.0,2.0,1,7,4.008861
2,38,33,6,motif,6,555,2,8,1.0,2.0,2,8,3.702315
3,38,33,6,motif,6,555,2,8,1.0,2.0,3,9,2.388741
4,38,33,6,motif,6,555,2,8,1.0,2.0,4,10,2.788471


In [8]:
# ============================================================
# BEST GLOBAL-TOP MATCH PER SHAPELET AND WINDOW LENGTH
# ============================================================

best_matches = []


for shapelet_index in sorted(
    top_pattern_comparison_df[
        "shapelet_index"
    ].unique()
):

    for w in WINDOWS:

        current = (
            top_pattern_comparison_df[
                (
                    top_pattern_comparison_df[
                        "shapelet_index"
                    ]
                    == shapelet_index
                )
                &
                (
                    top_pattern_comparison_df[
                        "w"
                    ]
                    == w
                )
            ]
        )


        if len(current) == 0:
            continue


        row = {
            "shapelet_index":
                shapelet_index,

            "w":
                w
        }


        # Best motif
        motifs = current[
            current[
                "pattern_type"
            ] == "motif"
        ]

        if len(motifs) > 0:

            best_motif = motifs.loc[
                motifs[
                    "distance"
                ].idxmin()
            ]

            row.update({
                "best_motif_distance":
                    float(
                        best_motif[
                            "distance"
                        ]
                    ),

                "best_motif_global_rank":
                    int(
                        best_motif[
                            "global_pattern_rank"
                        ]
                    ),

                "best_motif_ts_index":
                    int(
                        best_motif[
                            "pattern_ts_index"
                        ]
                    ),

                "best_motif_start":
                    int(
                        best_motif[
                            "pattern_start"
                        ]
                    ),

                "best_motif_shapelet_window_start":
                    int(
                        best_motif[
                            "shapelet_window_start"
                        ]
                    )
            })


        # Best discord
        discords = current[
            current[
                "pattern_type"
            ] == "discord"
        ]

        if len(discords) > 0:

            best_discord = discords.loc[
                discords[
                    "distance"
                ].idxmin()
            ]

            row.update({
                "best_discord_distance":
                    float(
                        best_discord[
                            "distance"
                        ]
                    ),

                "best_discord_global_rank":
                    int(
                        best_discord[
                            "global_pattern_rank"
                        ]
                    ),

                "best_discord_ts_index":
                    int(
                        best_discord[
                            "pattern_ts_index"
                        ]
                    ),

                "best_discord_start":
                    int(
                        best_discord[
                            "pattern_start"
                        ]
                    ),

                "best_discord_shapelet_window_start":
                    int(
                        best_discord[
                            "shapelet_window_start"
                        ]
                    )
            })


        best_matches.append(
            row
        )


global_top_match_df = pd.DataFrame(
    best_matches
)

In [9]:
# ============================================================
# CLOSEST TOP PATTERN TYPE
# ============================================================

global_top_match_df[
    "closest_pattern_type"
] = np.where(
    global_top_match_df[
        "best_motif_distance"
    ]
    <
    global_top_match_df[
        "best_discord_distance"
    ],
    "motif",
    "discord"
)


global_top_match_df = (
    global_top_match_df
    .merge(
        shapelet_metadata_df[
            [
                "shapelet_index",
                "shapelet_length",
                "abs_lr_coefficient"
            ]
        ],
        on="shapelet_index",
        how="left"
    )
)


global_top_match_df = (
    global_top_match_df
    .sort_values(
        [
            "shapelet_index",
            "w"
        ]
    )
    .reset_index(drop=True)
)


display(
    global_top_match_df
)

,shapelet_index,w,best_motif_distance,best_motif_global_rank,best_motif_ts_index,best_motif_start,best_motif_shapelet_window_start,best_discord_distance,best_discord_global_rank,best_discord_ts_index,best_discord_start,best_discord_shapelet_window_start,closest_pattern_type,shapelet_length,abs_lr_coefficient
0,0,6,1.574414,4,759,48,13,0.815957,2,2773,181,10,discord,23,0.770118
1,0,12,2.918936,3,1987,123,2,2.145254,8,1651,43,6,discord,23,0.770118
2,15,6,1.005647,9,2869,127,0,1.838766,5,1983,106,4,motif,15,0.954442
3,15,12,2.201893,10,456,95,0,2.790856,6,1732,102,3,motif,15,0.954442
4,21,6,1.308736,9,2869,127,5,1.823101,4,3501,104,6,motif,14,0.846139
5,21,12,2.653965,6,3506,179,1,4.059737,4,1794,167,0,motif,14,0.846139
6,26,6,1.367475,9,2869,14,15,0.846175,4,3501,104,18,discord,24,0.845975
7,26,12,2.821110,6,3506,179,0,2.825527,6,1732,102,11,motif,24,0.845975
8,30,6,0.394152,10,1192,64,18,1.896478,2,2773,181,16,motif,35,0.850050
9,30,12,1.069757,3,1987,95,1,3.079349,8,1651,43,7,motif,35,0.850050


In [10]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print("=" * 80)
print("SHAPELETS vs GLOBAL TOP MOTIFS / DISCORDS")
print("=" * 80)


for w in WINDOWS:

    current = (
        global_top_match_df[
            global_top_match_df[
                "w"
            ] == w
        ]
    )

    print(
        f"\nWINDOW LENGTH w = {w}"
    )

    print(
        "\nClosest pattern type:"
    )

    print(
        current[
            "closest_pattern_type"
        ]
        .value_counts()
    )

    print(
        "\nMean best motif distance:",
        current[
            "best_motif_distance"
        ].mean()
    )

    print(
        "Mean best discord distance:",
        current[
            "best_discord_distance"
        ].mean()
    )

    print(
        "\nMedian best motif distance:",
        current[
            "best_motif_distance"
        ].median()
    )

    print(
        "Median best discord distance:",
        current[
            "best_discord_distance"
        ].median()
    )


print("\n" + "=" * 80)
print("OVERALL ACROSS BOTH WINDOW LENGTHS")
print("=" * 80)

print(
    global_top_match_df[
        "closest_pattern_type"
    ]
    .value_counts()
)

SHAPELETS vs GLOBAL TOP MOTIFS / DISCORDS

WINDOW LENGTH w = 6

Closest pattern type:
closest_pattern_type
motif      8
discord    2
Name: count, dtype: int64

Mean best motif distance: 1.0235524809282635
Mean best discord distance: 1.587592852512718

Median best motif distance: 1.0522071272991966
Median best discord distance: 1.8014280711810486

WINDOW LENGTH w = 12

Closest pattern type:
closest_pattern_type
motif      9
discord    1
Name: count, dtype: int64

Mean best motif distance: 2.202926166159806
Mean best discord distance: 3.1917738957748973

Median best motif distance: 2.136492035617248
Median best discord distance: 2.979818807033025

OVERALL ACROSS BOTH WINDOW LENGTHS
closest_pattern_type
motif      17
discord     3
Name: count, dtype: int64


In [12]:
# ============================================================
# BUILD ALL MOTIF / DISCORD PATTERN BANKS
# Same representation used in the TOP-10 comparison
# ============================================================

import numpy as np
import pandas as pd

WINDOWS = [6, 12]


def extract_pattern_values(ts_index, start, w):
    """
    Extract a length-w ENMO subsequence from the original time series
    and apply local z-normalization.
    """
    values = (
        df[int(ts_index)]["enmo"]
        .to_numpy(dtype=float)[
            int(start):int(start) + int(w)
        ]
    )

    if len(values) != w:
        return None

    return z_normalize(values)


all_motif_banks = {}
all_discord_banks = {}

all_motif_metadata = {}
all_discord_metadata = {}


for w in WINDOWS:

    # ========================================================
    # ALL MOTIF OCCURRENCES
    # ========================================================

    motif_arrays = []
    motif_meta = []

    current_motifs = motif_occurrences_df[
        motif_occurrences_df["w"] == w
    ]

    for _, row in current_motifs.iterrows():

        values = extract_pattern_values(
            ts_index=row["ts_index"],
            start=row["motif_start"],
            w=w
        )

        if values is None:
            continue

        motif_arrays.append(values)

        motif_meta.append({
            "ts_index":
                int(row["ts_index"]),

            "w":
                w,

            "motif_rank":
                int(row["motif_rank"]),

            "motif_start":
                int(row["motif_start"])
        })


    all_motif_banks[w] = np.vstack(
        motif_arrays
    )

    all_motif_metadata[w] = pd.DataFrame(
        motif_meta
    )


    # ========================================================
    # ALL DISCORDS
    # ========================================================

    discord_arrays = []
    discord_meta = []

    current_discords = discords_df[
        discords_df["w"] == w
    ]

    for _, row in current_discords.iterrows():

        values = extract_pattern_values(
            ts_index=row["ts_index"],
            start=row["discord_start"],
            w=w
        )

        if values is None:
            continue

        discord_arrays.append(values)

        discord_meta.append({
            "ts_index":
                int(row["ts_index"]),

            "w":
                w,

            "discord_rank":
                int(row["discord_rank"]),

            "discord_start":
                int(row["discord_start"]),

            "discord_score":
                float(row["discord_score"])
        })


    all_discord_banks[w] = np.vstack(
        discord_arrays
    )

    all_discord_metadata[w] = pd.DataFrame(
        discord_meta
    )


    print(
        f"w={w}: "
        f"{len(all_motif_banks[w]):,} motif occurrences, "
        f"{len(all_discord_banks[w]):,} discords"
    )

w=6: 27,688 motif occurrences, 12,765 discords
w=12: 43,423 motif occurrences, 12,771 discords


In [13]:
# ============================================================
# FIND CLOSEST PATTERN IN A COMPLETE PATTERN BANK
# ============================================================

def find_best_match_in_bank(
    shapelet_values,
    w,
    pattern_bank
):
    """
    Compare every length-w window inside a shapelet with every
    pattern in the supplied bank.

    All sequences are locally z-normalized.

    Returns the globally minimum Euclidean distance and the
    indices of the corresponding shapelet window and pattern.
    """

    shapelet_windows = get_shapelet_windows(
        shapelet_values,
        w
    )

    if len(shapelet_windows) == 0:
        return None


    # Matrix:
    # number of shapelet windows x w
    S = np.vstack([
        sw["values"]
        for sw in shapelet_windows
    ])


    best_distance = np.inf
    best_shapelet_window_index = None
    best_pattern_index = None


    # Process one shapelet window at a time.
    # This avoids creating unnecessarily large 3D arrays.
    for i, sw in enumerate(S):

        distances = np.linalg.norm(
            pattern_bank - sw,
            axis=1
        )

        j = int(
            np.argmin(distances)
        )

        d = float(
            distances[j]
        )


        if d < best_distance:

            best_distance = d

            best_shapelet_window_index = i

            best_pattern_index = j


    best_sw = shapelet_windows[
        best_shapelet_window_index
    ]


    return {
        "distance":
            best_distance,

        "shapelet_window_start":
            int(
                best_sw[
                    "shapelet_window_start"
                ]
            ),

        "shapelet_window_end":
            int(
                best_sw[
                    "shapelet_window_end"
                ]
            ),

        "pattern_index":
            int(
                best_pattern_index
            )
    }

In [14]:
# ============================================================
# SHAPELETS vs ALL EXTRACTED MOTIFS AND DISCORDS
# ============================================================

all_pattern_results = []


for _, shp in shapelet_metadata_df.iterrows():

    shapelet_index = int(
        shp["shapelet_index"]
    )

    shapelet_values = np.asarray(
        shp["shapelet_values"],
        dtype=float
    ).ravel()

    shapelet_length = len(
        shapelet_values
    )


    for w in WINDOWS:

        if shapelet_length < w:
            continue


        # ====================================================
        # CLOSEST MOTIF AMONG ALL MOTIF OCCURRENCES
        # ====================================================

        best_motif = find_best_match_in_bank(
            shapelet_values=
                shapelet_values,

            w=
                w,

            pattern_bank=
                all_motif_banks[w]
        )


        # ====================================================
        # CLOSEST DISCORD AMONG ALL DISCORDS
        # ====================================================

        best_discord = find_best_match_in_bank(
            shapelet_values=
                shapelet_values,

            w=
                w,

            pattern_bank=
                all_discord_banks[w]
        )


        row = {
            "shapelet_index":
                shapelet_index,

            "shapelet_length":
                shapelet_length,

            "w":
                w,

            "all_best_motif_distance":
                best_motif["distance"],

            "all_best_discord_distance":
                best_discord["distance"],

            "all_best_motif_shapelet_window_start":
                best_motif[
                    "shapelet_window_start"
                ],

            "all_best_discord_shapelet_window_start":
                best_discord[
                    "shapelet_window_start"
                ]
        }


        # Metadata of closest motif
        motif_meta = (
            all_motif_metadata[w]
            .iloc[
                best_motif[
                    "pattern_index"
                ]
            ]
        )

        row.update({
            "all_best_motif_ts_index":
                int(
                    motif_meta[
                        "ts_index"
                    ]
                ),

            "all_best_motif_rank":
                int(
                    motif_meta[
                        "motif_rank"
                    ]
                ),

            "all_best_motif_start":
                int(
                    motif_meta[
                        "motif_start"
                    ]
                )
        })


        # Metadata of closest discord
        discord_meta = (
            all_discord_metadata[w]
            .iloc[
                best_discord[
                    "pattern_index"
                ]
            ]
        )

        row.update({
            "all_best_discord_ts_index":
                int(
                    discord_meta[
                        "ts_index"
                    ]
                ),

            "all_best_discord_rank":
                int(
                    discord_meta[
                        "discord_rank"
                    ]
                ),

            "all_best_discord_start":
                int(
                    discord_meta[
                        "discord_start"
                    ]
                )
        })


        row[
            "all_closest_pattern_type"
        ] = (
            "motif"
            if
            row[
                "all_best_motif_distance"
            ]
            <
            row[
                "all_best_discord_distance"
            ]
            else
            "discord"
        )


        all_pattern_results.append(
            row
        )


all_pattern_match_df = pd.DataFrame(
    all_pattern_results
)


display(
    all_pattern_match_df
)

,shapelet_index,shapelet_length,w,all_best_motif_distance,all_best_discord_distance,all_best_motif_shapelet_window_start,all_best_discord_shapelet_window_start,all_best_motif_ts_index,all_best_motif_rank,all_best_motif_start,all_best_discord_ts_index,all_best_discord_rank,all_best_discord_start,all_closest_pattern_type
0,38,33,6,7.872609e-09,1.626915e-01,12,4,244,3,135,3014,3,177,motif
1,38,33,12,1.051816e-07,1.019634e+00,21,5,244,3,144,124,3,85,motif
2,15,15,6,1.464663e-01,2.643998e-01,0,6,413,2,89,433,3,106,motif
3,15,15,12,1.260187e+00,1.444973e+00,0,2,3685,3,96,1951,1,180,motif
4,30,35,6,3.857038e-02,9.474838e-02,18,9,2518,2,72,142,2,86,motif
5,30,35,12,4.805824e-08,9.950706e-01,19,10,1289,1,174,2247,2,80,motif
6,21,14,6,4.263447e-02,1.777553e-01,3,8,754,3,105,364,3,13,motif
7,21,14,12,7.208356e-01,1.533409e+00,1,0,3598,3,117,439,3,84,motif
8,26,24,6,9.252711e-08,2.643053e-07,7,16,868,3,63,868,3,72,motif
9,26,24,12,1.495137e+00,2.486426e-07,0,11,2426,3,107,868,1,67,discord


In [15]:
# ============================================================
# SUMMARY: ALL PATTERNS
# ============================================================

print("=" * 80)
print("SHAPELETS vs ALL EXTRACTED MOTIFS / DISCORDS")
print("=" * 80)


for w in WINDOWS:

    current = all_pattern_match_df[
        all_pattern_match_df["w"] == w
    ]


    print(
        f"\nWINDOW LENGTH w = {w}"
    )

    print(
        "\nClosest pattern type:"
    )

    print(
        current[
            "all_closest_pattern_type"
        ]
        .value_counts()
    )


    print(
        "\nMean closest motif distance:",
        current[
            "all_best_motif_distance"
        ].mean()
    )

    print(
        "Mean closest discord distance:",
        current[
            "all_best_discord_distance"
        ].mean()
    )


    print(
        "\nMedian closest motif distance:",
        current[
            "all_best_motif_distance"
        ].median()
    )

    print(
        "Median closest discord distance:",
        current[
            "all_best_discord_distance"
        ].median()
    )

SHAPELETS vs ALL EXTRACTED MOTIFS / DISCORDS

WINDOW LENGTH w = 6

Closest pattern type:
all_closest_pattern_type
motif    10
Name: count, dtype: int64

Mean closest motif distance: 0.06497432045761144
Mean closest discord distance: 0.17172760399373407

Median closest motif distance: 0.040602426557016374
Median closest discord distance: 0.17062131295733984

WINDOW LENGTH w = 12

Closest pattern type:
all_closest_pattern_type
motif      8
discord    2
Name: count, dtype: int64

Mean closest motif distance: 0.578822530037596
Mean closest discord distance: 1.1460027678323612

Median closest motif distance: 0.3604180415332326
Median closest discord distance: 1.4297408687569142


In [16]:
# ============================================================
# DIRECT COMPARISON:
# TOP-10 GLOBAL PATTERNS vs ALL EXTRACTED PATTERNS
# ============================================================

top10_vs_all_df = (
    global_top_match_df[
        [
            "shapelet_index",
            "w",
            "best_motif_distance",
            "best_discord_distance",
            "closest_pattern_type"
        ]
    ]
    .rename(
        columns={
            "best_motif_distance":
                "top10_best_motif_distance",

            "best_discord_distance":
                "top10_best_discord_distance",

            "closest_pattern_type":
                "top10_closest_pattern_type"
        }
    )
    .merge(
        all_pattern_match_df[
            [
                "shapelet_index",
                "w",
                "all_best_motif_distance",
                "all_best_discord_distance",
                "all_closest_pattern_type"
            ]
        ],
        on=[
            "shapelet_index",
            "w"
        ],
        how="inner"
    )
)


# How much farther is the best TOP-10 pattern
# than the unrestricted closest pattern?

top10_vs_all_df[
    "motif_distance_gap"
] = (
    top10_vs_all_df[
        "top10_best_motif_distance"
    ]
    -
    top10_vs_all_df[
        "all_best_motif_distance"
    ]
)


top10_vs_all_df[
    "discord_distance_gap"
] = (
    top10_vs_all_df[
        "top10_best_discord_distance"
    ]
    -
    top10_vs_all_df[
        "all_best_discord_distance"
    ]
)


display(
    top10_vs_all_df
)

,shapelet_index,w,top10_best_motif_distance,top10_best_discord_distance,top10_closest_pattern_type,all_best_motif_distance,all_best_discord_distance,all_closest_pattern_type,motif_distance_gap,discord_distance_gap
0,0,6,1.574414,0.815957,discord,1.608971e-01,2.369470e-01,motif,1.413517,0.579010
1,0,12,2.918936,2.145254,discord,1.378371e+00,4.643257e-07,discord,1.540565,2.145254
2,15,6,1.005647,1.838766,motif,1.464663e-01,2.643998e-01,motif,0.859181,1.574366
3,15,12,2.201893,2.790856,motif,1.260187e+00,1.444973e+00,motif,0.941706,1.345883
4,21,6,1.308736,1.823101,motif,4.263447e-02,1.777553e-01,motif,1.266102,1.645346
5,21,12,2.653965,4.059737,motif,7.208356e-01,1.533409e+00,motif,1.933129,2.526328
6,26,6,1.367475,0.846175,discord,9.252711e-08,2.643053e-07,motif,1.367475,0.846175
7,26,12,2.821110,2.825527,motif,1.495137e+00,2.486426e-07,discord,1.325973,2.825527
8,30,6,0.394152,1.896478,motif,3.857038e-02,9.474838e-02,motif,0.355582,1.801730
9,30,12,1.069757,3.079349,motif,4.805824e-08,9.950706e-01,motif,1.069757,2.084279


In [17]:
# ============================================================
# TOP-10 vs ALL SUMMARY
# ============================================================

print("=" * 80)
print("TOP-10 GLOBAL PATTERNS vs ALL EXTRACTED PATTERNS")
print("=" * 80)


for w in WINDOWS:

    current = top10_vs_all_df[
        top10_vs_all_df["w"] == w
    ]


    print(
        f"\nWINDOW LENGTH w = {w}"
    )


    print("\nMOTIFS")

    print(
        "Mean TOP-10 best distance:",
        current[
            "top10_best_motif_distance"
        ].mean()
    )

    print(
        "Mean ALL-pattern best distance:",
        current[
            "all_best_motif_distance"
        ].mean()
    )

    print(
        "Mean distance gap:",
        current[
            "motif_distance_gap"
        ].mean()
    )


    print("\nDISCORDS")

    print(
        "Mean TOP-10 best distance:",
        current[
            "top10_best_discord_distance"
        ].mean()
    )

    print(
        "Mean ALL-pattern best distance:",
        current[
            "all_best_discord_distance"
        ].mean()
    )

    print(
        "Mean distance gap:",
        current[
            "discord_distance_gap"
        ].mean()
    )


    print(
        "\nClosest type using TOP-10:"
    )

    print(
        current[
            "top10_closest_pattern_type"
        ]
        .value_counts()
    )


    print(
        "\nClosest type using ALL patterns:"
    )

    print(
        current[
            "all_closest_pattern_type"
        ]
        .value_counts()
    )

TOP-10 GLOBAL PATTERNS vs ALL EXTRACTED PATTERNS

WINDOW LENGTH w = 6

MOTIFS
Mean TOP-10 best distance: 1.0235524809282635
Mean ALL-pattern best distance: 0.06497432045761144
Mean distance gap: 0.9585781604706523

DISCORDS
Mean TOP-10 best distance: 1.587592852512718
Mean ALL-pattern best distance: 0.17172760399373407
Mean distance gap: 1.415865248518984

Closest type using TOP-10:
top10_closest_pattern_type
motif      8
discord    2
Name: count, dtype: int64

Closest type using ALL patterns:
all_closest_pattern_type
motif    10
Name: count, dtype: int64

WINDOW LENGTH w = 12

MOTIFS
Mean TOP-10 best distance: 2.202926166159806
Mean ALL-pattern best distance: 0.578822530037596
Mean distance gap: 1.6241036361222096

DISCORDS
Mean TOP-10 best distance: 3.1917738957748973
Mean ALL-pattern best distance: 1.1460027678323612
Mean distance gap: 2.045771127942536

Closest type using TOP-10:
top10_closest_pattern_type
motif      9
discord    1
Name: count, dtype: int64

Closest type using ALL 

## Save final relationship outputs

In [18]:
global_top_match_df.to_csv(
    COMPARE_DIR/"top10_shapelet_vs_top10_motif_discord.csv",index=False
)
all_pattern_match_df.to_csv(
    COMPARE_DIR/"shapelet_vs_all_motif_discord.csv",index=False
)
top10_vs_all_df.to_csv(
    COMPARE_DIR/"top10_vs_all_pattern_comparison.csv",index=False
)

summary_rows = []
for w in WINDOWS:
    top = global_top_match_df[global_top_match_df["w"]==w]
    allp = all_pattern_match_df[all_pattern_match_df["w"]==w]

    summary_rows.append({
        "w":w,
        "top10_motif_closest_count":int((top["closest_pattern_type"]=="motif").sum()),
        "top10_discord_closest_count":int((top["closest_pattern_type"]=="discord").sum()),
        "top10_mean_motif_distance":top["best_motif_distance"].mean(),
        "top10_mean_discord_distance":top["best_discord_distance"].mean(),
        "top10_median_motif_distance":top["best_motif_distance"].median(),
        "top10_median_discord_distance":top["best_discord_distance"].median(),
        "all_motif_closest_count":int((allp["all_closest_pattern_type"]=="motif").sum()),
        "all_discord_closest_count":int((allp["all_closest_pattern_type"]=="discord").sum()),
        "all_mean_motif_distance":allp["all_best_motif_distance"].mean(),
        "all_mean_discord_distance":allp["all_best_discord_distance"].mean(),
    })

relationship_summary_df = pd.DataFrame(summary_rows)
display(relationship_summary_df)
relationship_summary_df.to_csv(
    COMPARE_DIR/"shapelet_relation_summary.csv",index=False
)

print("Final comparison artifacts saved in:",COMPARE_DIR)

,w,top10_motif_closest_count,top10_discord_closest_count,top10_mean_motif_distance,top10_mean_discord_distance,top10_median_motif_distance,top10_median_discord_distance,all_motif_closest_count,all_discord_closest_count,all_mean_motif_distance,all_mean_discord_distance
0,6,8,2,1.023552,1.587593,1.052207,1.801428,10,0,0.064974,0.171728
1,12,9,1,2.202926,3.191774,2.136492,2.979819,8,2,0.578823,1.146003


Final comparison artifacts saved in: /Users/sarabiiavasco/Codice/DATA MINING/PROGETTO DM2/artifacts/module3/shapelet_motif_discord_comparison


## Final interpretation

The project conclusion is based on the **actual distances computed above**, not on a visual impression.

In the historical final run:
- for both `w=6` and `w=12`, 9 of the 10 selected shapelets were closer to a global top motif than to a global top discord;
- against all extracted patterns, motifs were closest for 7/10 shapelets at `w=6` and 9/10 at `w=12`.

This supports the interpretation that the discriminative shapelets generally resemble **recurring behavioural patterns** more than extreme anomalous episodes, especially at the longer six-hour scale.